# DWS SnowCamp — Day 1: Data Engineering in Snowflake Workspaces

## From Raw Data to Production dbt Pipelines

**Duration**: ~1.5 hours  |  **Level**: Intermediate  |  **Account**: Snowflake Trial

---

Welcome to Day 1 of the DWS SnowCamp hands-on lab! Today you will experience what it is like to build a **production-grade data pipeline** for asset-management client reporting — entirely inside Snowflake.

We will start by generating realistic synthetic data (the kind of thing an asset manager like DWS would receive from trading systems and custodians), enrich it with **real NASDAQ market prices** from the Snowflake Marketplace, and then build a proper **dbt transformation pipeline** that turns messy source tables into clean, tested, business-ready data products.

By the end of today you will have:

| What | Why It Matters |
|------|---------------|
| A three-schema database (RAW → ANALYTICS → MARTS) | Mirrors how DWS separates Bronze / Silver / Gold data |
| Six synthetic source tables with real NASDAQ tickers | Realistic test data that joins with Marketplace prices |
| A full dbt project with staging, intermediate, and mart models | Automated, tested, version-controlled transformations |
| A scheduled Snowflake Task running dbt builds | Production-ready orchestration — no external scheduler needed |

Tomorrow in Day 2 we will pick up where we left off and build an interactive Streamlit dashboard, publish data products to the Internal Marketplace, and explore platform administration.

| Day 1 | Topic | Duration |
|-------|-------|----------|
| 1 | Environment Setup & Synthetic Data | 20 min |
| 2 | Snowflake Marketplace Integration | 10 min |
| 3 | dbt Transformation (Staging → Intermediate → Marts) | 30 min |
| 4 | dbt Projects: Deploy, Test & Orchestrate | 30 min |

> **Reference**: [Snowflake Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/workspaces)

### Architecture Overview

Before we write any code, let's understand the architecture we are building. This three-layer pattern is the standard for modern data platforms — you will see it in production at DWS and most large financial institutions.

```
Source Data (Python)  →  RAW schema  →  ANALYTICS schema  →  MARTS schema
                          (Bronze)        (Silver)              (Gold)
                                                                  ↓
                                                          Streamlit App (Day 2)
                                                                  ↓
                                                        Marketplace Listing (Day 2)

Snowflake Marketplace  →  ANALYTICS.STG_MARKET_PRICES  →  MARTS.F_HOLDINGS_WITH_MARKET_DATA
(Real NASDAQ prices)        (Staging view)                    (Joined with synthetic holdings)
```

**Why three layers?** Each layer serves a distinct purpose:

| Layer | Schema | What Happens Here |
|-------|--------|-------------------|
| **Bronze / Raw** | `RAW` | Data lands here exactly as the source system produces it. No transformations — just a faithful copy. |
| **Silver / Curated** | `ANALYTICS` | Staging views clean column names and types. Intermediate tables join across sources and add calculated fields. |
| **Gold / Marts** | `MARTS` | Business-ready fact and dimension tables. These are what analysts, dashboards, and downstream consumers query. |
| **Marketplace** | `FINANCIAL__ECONOMIC_ESSENTIALS` | Real NASDAQ prices via Snowflake's zero-copy data sharing — no ETL, no storage cost. |

In a production DWS deployment, raw data would arrive via [OpenFlow](https://docs.snowflake.com/en/user-guide/data-load-openflow) from on-prem Oracle and SQL Server databases. Today we simulate that with Python.

---
## Part 1: Environment Setup & Synthetic Data (20 min)

Let's start by creating our working environment. Every Snowflake project needs a **database** (logical container for all objects), **schemas** (to organise tables into layers), and a **warehouse** (compute engine that runs our queries).

Think of the warehouse like a cluster of servers that Snowflake spins up on demand and suspends when idle — you only pay for the seconds of compute you actually use. For this lab we use a `MEDIUM` warehouse to keep things responsive.

> **Reference**: [CREATE DATABASE](https://docs.snowflake.com/en/sql-reference/sql/create-database) | [CREATE WAREHOUSE](https://docs.snowflake.com/en/sql-reference/sql/create-warehouse)

In [ ]:
-- =============================================================
-- Create our lab database with three schemas (one per layer)
-- =============================================================

CREATE DATABASE IF NOT EXISTS SNOWCAMP_LAB;

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.RAW
    COMMENT = 'Raw/Bronze layer - synthetic source data';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.ANALYTICS
    COMMENT = 'Staging and intermediate layer - cleaned and enriched';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.MARTS
    COMMENT = 'Gold/Mart layer - business-ready data products';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.GITREPO
    COMMENT = 'Git repository integrations';

CREATE SCHEMA IF NOT EXISTS SNOWCAMP_LAB.NOTEBOOKS
    COMMENT = 'Snowflake Notebooks';

CREATE WAREHOUSE IF NOT EXISTS WH_LAB
    WAREHOUSE_SIZE   = 'MEDIUM'
    AUTO_SUSPEND     = 60
    AUTO_RESUME      = TRUE
    COMMENT          = 'SnowCamp hands-on lab warehouse';

ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

-- External access integration so the Workspaces compute service can reach the open internet
CREATE OR REPLACE NETWORK RULE snowcamp_egress_rule
    MODE       = EGRESS
    TYPE       = HOST_PORT
    VALUE_LIST = ('0.0.0.0:443', '0.0.0.0:80');

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION snowcamp_external_access
    ALLOWED_NETWORK_RULES         = (snowcamp_egress_rule)
    ALLOWED_AUTHENTICATION_SECRETS = ()
    ENABLED                        = TRUE;

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WH_LAB;
USE DATABASE SNOWCAMP_LAB;
USE SCHEMA RAW;

SELECT 'Environment ready!' AS status;

### Synthetic Data Model

Now let's populate the RAW schema with realistic asset-management data. In the real world, these tables would be fed by trading systems, portfolio accounting platforms, and market data vendors. For our lab we generate them with Python — but the important thing is that we use **real NASDAQ tickers** (AAPL, MSFT, NVDA, etc.) so that later we can join with actual market prices from the Snowflake Marketplace. That join is what makes this lab feel real.

Here is what we will create:

| Table | What It Represents | Approx. Rows |
|-------|-------------------|-------------|
| `CLIENTS` | Institutional investors (pension funds, insurers, etc.) | 30 |
| `PORTFOLIOS` | Investment portfolios, each linked to a client | 100 |
| `SECURITIES` | Equities and ETFs traded on NASDAQ | 50 |
| `HOLDINGS` | Daily position snapshots — "who holds what, and how much" | ~5,000 |
| `TRANSACTIONS` | Trade executions (buys and sells) | 10,000 |
| `BENCHMARKS` | Daily returns for benchmark indices (S&P 500, MSCI World, etc.) | ~1,200 |

For an asset manager like DWS, these six tables are the backbone of client reporting — every regulatory filing, performance report, and risk calculation starts from data like this.

> **Reference**: [Snowpark Python Developer Guide](https://docs.snowflake.com/en/developer-guide/snowpark/python/index)

In [ ]:
# ================================================================
# Generate synthetic asset-management data with REAL NASDAQ tickers
# ================================================================
import pandas as pd
import random
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session

session = get_active_session()
random.seed(42)

TICKER_DATA = [
    ('AAPL',  'Apple Inc',                   'Technology',              'Equity',  'USD'),
    ('MSFT',  'Microsoft Corp',              'Technology',              'Equity',  'USD'),
    ('AMZN',  'Amazon.com Inc',              'Consumer Discretionary',  'Equity',  'USD'),
    ('NVDA',  'NVIDIA Corp',                 'Technology',              'Equity',  'USD'),
    ('GOOGL', 'Alphabet Inc Class A',        'Communication Services',  'Equity',  'USD'),
    ('META',  'Meta Platforms Inc',          'Communication Services',  'Equity',  'USD'),
    ('TSLA',  'Tesla Inc',                   'Consumer Discretionary',  'Equity',  'USD'),
    ('JPM',   'JPMorgan Chase & Co',         'Financials',              'Equity',  'USD'),
    ('V',     'Visa Inc',                    'Financials',              'Equity',  'USD'),
    ('JNJ',   'Johnson & Johnson',           'Healthcare',              'Equity',  'USD'),
    ('UNH',   'UnitedHealth Group Inc',      'Healthcare',              'Equity',  'USD'),
    ('HD',    'Home Depot Inc',              'Consumer Discretionary',  'Equity',  'USD'),
    ('PG',    'Procter & Gamble Co',         'Consumer Discretionary',  'Equity',  'USD'),
    ('MA',    'Mastercard Inc',              'Financials',              'Equity',  'USD'),
    ('XOM',   'Exxon Mobil Corp',            'Energy',                  'Equity',  'USD'),
    ('CVX',   'Chevron Corp',                'Energy',                  'Equity',  'USD'),
    ('LLY',   'Eli Lilly and Co',            'Healthcare',              'Equity',  'USD'),
    ('ABBV',  'AbbVie Inc',                  'Healthcare',              'Equity',  'USD'),
    ('PFE',   'Pfizer Inc',                  'Healthcare',              'Equity',  'USD'),
    ('MRK',   'Merck & Co Inc',              'Healthcare',              'Equity',  'USD'),
    ('COST',  'Costco Wholesale Corp',       'Consumer Discretionary',  'Equity',  'USD'),
    ('WMT',   'Walmart Inc',                 'Consumer Discretionary',  'Equity',  'USD'),
    ('DIS',   'Walt Disney Co',              'Communication Services',  'Equity',  'USD'),
    ('NFLX',  'Netflix Inc',                 'Communication Services',  'Equity',  'USD'),
    ('ADBE',  'Adobe Inc',                   'Technology',              'Equity',  'USD'),
    ('CRM',   'Salesforce Inc',              'Technology',              'Equity',  'USD'),
    ('AMD',   'Advanced Micro Devices Inc',  'Technology',              'Equity',  'USD'),
    ('INTC',  'Intel Corp',                  'Technology',              'Equity',  'USD'),
    ('NEE',   'NextEra Energy Inc',          'Utilities',               'Equity',  'USD'),
    ('DUK',   'Duke Energy Corp',            'Utilities',               'Equity',  'USD'),
    ('SO',    'Southern Co',                 'Utilities',               'Equity',  'USD'),
    ('BLK',   'BlackRock Inc',               'Financials',              'Equity',  'USD'),
    ('GS',    'Goldman Sachs Group Inc',     'Financials',              'Equity',  'USD'),
    ('MS',    'Morgan Stanley',              'Financials',              'Equity',  'USD'),
    ('AMT',   'American Tower Corp',         'Real Estate',             'Equity',  'USD'),
    ('PLD',   'Prologis Inc',                'Real Estate',             'Equity',  'USD'),
    ('SPY',   'SPDR S&P 500 ETF',           'Financials',              'ETF',     'USD'),
    ('QQQ',   'Invesco QQQ Trust',           'Technology',              'ETF',     'USD'),
    ('IWM',   'iShares Russell 2000 ETF',    'Financials',              'ETF',     'USD'),
    ('EFA',   'iShares MSCI EAFE ETF',       'Financials',              'ETF',     'USD'),
    ('VWO',   'Vanguard FTSE Emerging Mkts', 'Financials',              'ETF',     'USD'),
    ('AGG',   'iShares Core US Aggregate',   'Financials',              'ETF',     'USD'),
    ('TLT',   'iShares 20+ Year Treasury',   'Financials',              'ETF',     'USD'),
    ('LQD',   'iShares Investment Grade',    'Financials',              'ETF',     'USD'),
    ('GLD',   'SPDR Gold Shares',            'Materials',               'ETF',     'USD'),
    ('CAT',   'Caterpillar Inc',             'Industrials',             'Equity',  'USD'),
    ('BA',    'Boeing Co',                   'Industrials',             'Equity',  'USD'),
    ('HON',   'Honeywell International',     'Industrials',             'Equity',  'USD'),
    ('LIN',   'Linde PLC',                   'Materials',               'Equity',  'USD'),
    ('FCX',   'Freeport-McMoRan Inc',        'Materials',               'Equity',  'USD'),
]

NUM_CLIENTS       = 30
NUM_PORTFOLIOS    = 100
NUM_HOLDING_DAYS  = 20
NUM_TRANSACTIONS  = 10000
BENCHMARK_DAYS    = 252

REGIONS         = ['EMEA', 'APAC', 'Americas']
AUM_TIERS       = ['Tier 1 (>1B)', 'Tier 2 (500M-1B)', 'Tier 3 (<500M)']
PORTFOLIO_TYPES = ['Equity', 'Fixed Income', 'Multi-Asset', 'ESG', 'Alternatives']
BENCHMARKS      = {'BM001': 'MSCI World', 'BM002': 'Euro Stoxx 50',
                   'BM003': 'S&P 500', 'BM004': 'Bloomberg Global Agg',
                   'BM005': 'FTSE 100'}
FIRST_NAMES     = ['Anna', 'Hans', 'Maria', 'Klaus', 'Sophie', 'Thomas',
                   'Emma', 'Felix', 'Laura', 'Max', 'Julia', 'Stefan',
                   'Lisa', 'Andreas', 'Sarah']
LAST_NAMES      = ['Mueller', 'Schmidt', 'Fischer', 'Weber', 'Meyer',
                   'Wagner', 'Becker', 'Schulz', 'Hoffmann', 'Koch',
                   'Richter', 'Klein', 'Wolf', 'Braun', 'Lange']
COMPANY_SUFFIX  = ['Capital', 'Invest', 'Advisors', 'Partners',
                   'Asset Mgmt', 'Wealth', 'Fund Services']

clients = []
for i in range(1, NUM_CLIENTS + 1):
    name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)} {random.choice(COMPANY_SUFFIX)}"
    clients.append({
        'CLIENT_ID': f'CLT{i:04d}', 'CLIENT_NAME': name,
        'REGION': random.choice(REGIONS), 'AUM_TIER': random.choice(AUM_TIERS),
        'ONBOARDING_DATE': (datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1500))).strftime('%Y-%m-%d')
    })

portfolios = []
for i in range(1, NUM_PORTFOLIOS + 1):
    client = random.choice(clients)
    portfolios.append({
        'PORTFOLIO_ID': f'PF{i:04d}', 'CLIENT_ID': client['CLIENT_ID'],
        'PORTFOLIO_NAME': f"{client['CLIENT_NAME'].split()[0]} {random.choice(PORTFOLIO_TYPES)} Fund {i}",
        'PORTFOLIO_TYPE': random.choice(PORTFOLIO_TYPES),
        'BENCHMARK_ID': random.choice(list(BENCHMARKS.keys())),
        'INCEPTION_DATE': (datetime(2019, 1, 1) + timedelta(days=random.randint(0, 1800))).strftime('%Y-%m-%d')
    })

securities = []
for i, (ticker, name, sector, asset_class, ccy) in enumerate(TICKER_DATA, 1):
    securities.append({
        'SECURITY_ID': f'SEC{i:04d}', 'TICKER': ticker,
        'ISIN': f'US{random.randint(1000000000, 9999999999)}',
        'SECURITY_NAME': name, 'SECTOR': sector,
        'ASSET_CLASS': asset_class, 'CURRENCY': ccy
    })

holdings = []
hid = 1
base_date = datetime(2025, 1, 6)
for day_offset in range(NUM_HOLDING_DAYS):
    d = base_date + timedelta(days=day_offset)
    if d.weekday() >= 5:
        continue
    for pf in random.sample(portfolios, min(50, len(portfolios))):
        for sec in random.sample(securities, random.randint(3, 8)):
            qty = random.randint(100, 50000)
            price = round(random.uniform(10, 500), 2)
            holdings.append({
                'HOLDING_ID': hid, 'PORTFOLIO_ID': pf['PORTFOLIO_ID'],
                'SECURITY_ID': sec['SECURITY_ID'],
                'AS_OF_DATE': d.strftime('%Y-%m-%d'),
                'QUANTITY': qty, 'MARKET_VALUE': round(qty * price, 2)
            })
            hid += 1

transactions = []
for i in range(1, NUM_TRANSACTIONS + 1):
    pf = random.choice(portfolios)
    sec = random.choice(securities)
    side = random.choice(['BUY', 'SELL'])
    qty = random.randint(50, 10000)
    price = round(random.uniform(10, 500), 2)
    td = base_date + timedelta(days=random.randint(0, NUM_HOLDING_DAYS - 1))
    transactions.append({
        'TRANSACTION_ID': f'TXN{i:06d}', 'PORTFOLIO_ID': pf['PORTFOLIO_ID'],
        'SECURITY_ID': sec['SECURITY_ID'], 'TRADE_DATE': td.strftime('%Y-%m-%d'),
        'SIDE': side, 'QUANTITY': qty, 'PRICE': price, 'AMOUNT': round(qty * price, 2)
    })

benchmarks = []
for bm_id, bm_name in BENCHMARKS.items():
    for day_offset in range(BENCHMARK_DAYS):
        d = datetime(2024, 2, 1) + timedelta(days=day_offset)
        if d.weekday() >= 5:
            continue
        benchmarks.append({
            'BENCHMARK_ID': bm_id, 'BENCHMARK_NAME': bm_name,
            'AS_OF_DATE': d.strftime('%Y-%m-%d'),
            'DAILY_RETURN': round(random.gauss(0.0003, 0.012), 6)
        })

tables = {
    'CLIENTS': pd.DataFrame(clients),
    'PORTFOLIOS': pd.DataFrame(portfolios),
    'SECURITIES': pd.DataFrame(securities),
    'HOLDINGS': pd.DataFrame(holdings),
    'TRANSACTIONS': pd.DataFrame(transactions),
    'BENCHMARKS': pd.DataFrame(benchmarks),
}
for name, df in tables.items():
    session.write_pandas(df, name, auto_create_table=True, overwrite=True,
                         database='SNOWCAMP_LAB', schema='RAW')
    print(f"  {name}: {len(df):,} rows")

print("\nAll synthetic data loaded into SNOWCAMP_LAB.RAW!")

In [ ]:
-- Validate the raw data — every table should have rows
SELECT 'CLIENTS'      AS table_name, COUNT(*) AS row_count FROM SNOWCAMP_LAB.RAW.CLIENTS
UNION ALL SELECT 'PORTFOLIOS',   COUNT(*) FROM SNOWCAMP_LAB.RAW.PORTFOLIOS
UNION ALL SELECT 'SECURITIES',   COUNT(*) FROM SNOWCAMP_LAB.RAW.SECURITIES
UNION ALL SELECT 'HOLDINGS',     COUNT(*) FROM SNOWCAMP_LAB.RAW.HOLDINGS
UNION ALL SELECT 'TRANSACTIONS', COUNT(*) FROM SNOWCAMP_LAB.RAW.TRANSACTIONS
UNION ALL SELECT 'BENCHMARKS',   COUNT(*) FROM SNOWCAMP_LAB.RAW.BENCHMARKS
ORDER BY table_name;

In [ ]:
-- Take a peek at the securities — notice these are real NASDAQ tickers
SELECT security_id, ticker, security_name, sector, asset_class
FROM SNOWCAMP_LAB.RAW.SECURITIES
ORDER BY ticker
LIMIT 10;

---
## Part 2: Snowflake Marketplace Integration (10 min)

This is one of the most powerful capabilities in Snowflake: the **Marketplace** lets you bring in third-party datasets with **zero ETL and zero storage cost**. The data provider shares a read-only snapshot of their tables directly into your account — Snowflake calls this "zero-copy data sharing."

For our lab, we will install the **Snowflake Public Data (Free)** listing, which includes real NASDAQ stock prices updated daily. Because we used real tickers in our synthetic data (AAPL, MSFT, NVDA, etc.), we can join our holdings with actual market prices to calculate mark-to-market valuations.

This is exactly the workflow an asset manager would follow: combine proprietary portfolio data with market data from an external vendor — except here it happens with a single SQL JOIN instead of a complex data pipeline.

### Install Snowflake Public Data (Free)

Follow the step-by-step instructions in the **Hands-On Lab Instructions** (Step 3b) to install the **Snowflake Public Data (Free)** listing from the Snowflake Marketplace. The instructions include screenshots to guide you through the process.

Once installed, the database `FINANCIAL__ECONOMIC_ESSENTIALS` will appear in your account with **no storage cost** (zero-copy data sharing). Then return here and continue with the next cell.

> **Reference**: [Snowflake Marketplace](https://docs.snowflake.com/en/user-guide/data-marketplace)

In [ ]:
-- Let's verify the Marketplace data is accessible
-- You should see Apple's closing prices for recent trading days
SELECT *
FROM FINANCIAL__ECONOMIC_ESSENTIALS.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
WHERE ticker = 'AAPL'
  AND variable = 'post-market_close_adjusted'
ORDER BY date DESC
LIMIT 10;

---
## Part 3: dbt Transformation — Staging → Intermediate → Marts (30 min)

Now we get to the heart of data engineering: transforming raw source data into clean, reliable, business-ready tables. We will follow the **three-layer dbt pattern** that has become the industry standard.

Why does this matter for an asset manager? Because regulators, clients, and risk teams all need to trust the numbers. A well-structured transformation pipeline — with tests, lineage, and version control — gives you that trust. Every table in the mart layer can be traced back to its source, every column has been validated, and every change is tracked in Git.

Let's build each layer step by step, starting with staging.

| Layer | What Happens | Materialisation | Why |
|-------|-------------|-----------------|-----|
| **Staging** | Clean, rename, type-cast raw sources | Views | Cheap (no storage), always up-to-date |
| **Intermediate** | Join across staging, calculate derived fields | Tables | Performance — downstream queries are fast |
| **Marts** | Business-ready facts and dimensions | Tables | Optimised for analysts and dashboards |

> **Reference**: [dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) | [CREATE VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-view)

### 3a. Staging Layer (Views)

Staging models are the **first line of defence** against messy source data. Their job is simple: rename columns to a consistent standard, cast types, and filter out obvious garbage — but **never** join across tables. Each staging model maps to exactly one source table.

We materialise staging models as **views** rather than tables because they don't need to store data — they just provide a clean interface over the raw tables. This means they are always up-to-date and cost zero storage.

We also create a staging view over the **Snowflake Marketplace** data to extract NASDAQ closing prices. Notice how the Marketplace data is accessed exactly like any other table — that's the magic of zero-copy sharing.

In [ ]:
USE SCHEMA SNOWCAMP_LAB.ANALYTICS;

-- Staging: Holdings — add a calculated unit price
CREATE OR REPLACE VIEW STG_HOLDINGS AS
SELECT
    holding_id, portfolio_id, security_id, as_of_date, quantity, market_value,
    CASE WHEN quantity > 0 THEN ROUND(market_value / quantity, 4) ELSE 0 END AS unit_price
FROM SNOWCAMP_LAB.RAW.HOLDINGS
WHERE market_value IS NOT NULL;

-- Staging: Transactions — add signed quantity (negative for sells)
CREATE OR REPLACE VIEW STG_TRANSACTIONS AS
SELECT
    transaction_id, portfolio_id, security_id, trade_date, side,
    quantity, price, amount,
    CASE side WHEN 'BUY' THEN quantity WHEN 'SELL' THEN -quantity ELSE 0 END AS signed_quantity
FROM SNOWCAMP_LAB.RAW.TRANSACTIONS
WHERE price > 0;

-- Staging: Securities — includes TICKER for the Marketplace join
CREATE OR REPLACE VIEW STG_SECURITIES AS
SELECT security_id, ticker, isin, security_name, sector, asset_class, currency
FROM SNOWCAMP_LAB.RAW.SECURITIES;

SELECT 'Core staging views created!' AS status;

In [ ]:
-- Staging: Marketplace closing prices (Snowflake Public Data)
-- This view reaches across to the shared Marketplace database —
-- no ETL, no copy, no storage cost.
CREATE OR REPLACE VIEW SNOWCAMP_LAB.ANALYTICS.STG_MARKET_PRICES AS
SELECT
    ticker,
    date           AS price_date,
    value          AS close_price
FROM FINANCIAL__ECONOMIC_ESSENTIALS.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
WHERE variable = 'post-market_close_adjusted'
  AND value IS NOT NULL;

-- Verify: a sample of real closing prices
SELECT ticker, price_date, close_price
FROM SNOWCAMP_LAB.ANALYTICS.STG_MARKET_PRICES
WHERE ticker IN ('AAPL', 'MSFT', 'NVDA')
ORDER BY ticker, price_date DESC
LIMIT 10;

### 3b. Intermediate Layer (Tables)

Now things get interesting. Intermediate models are where we **join across staging** and calculate derived metrics. Think of this as the "enrichment" step — we take the clean building blocks from staging and combine them into something more useful.

Here we join holdings with securities to get the ticker and sector for each position, and we calculate **portfolio weights** (what percentage of a portfolio's total value is allocated to each security). This is a critical metric for risk management — regulators want to know if a portfolio is too concentrated in any single name.

We materialise intermediate models as **tables** because downstream queries need fast access to the joined data.

In [ ]:
CREATE OR REPLACE TABLE SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS AS
WITH portfolio_totals AS (
    SELECT portfolio_id, as_of_date, SUM(market_value) AS total_portfolio_value
    FROM STG_HOLDINGS GROUP BY portfolio_id, as_of_date
)
SELECT
    h.holding_id, h.portfolio_id, h.security_id, h.as_of_date,
    h.quantity, h.market_value, h.unit_price,
    s.ticker, s.isin, s.security_name, s.sector, s.asset_class, s.currency,
    pt.total_portfolio_value,
    CASE WHEN pt.total_portfolio_value > 0
         THEN ROUND(h.market_value / pt.total_portfolio_value, 6) ELSE 0
    END AS portfolio_weight
FROM STG_HOLDINGS h
JOIN STG_SECURITIES s ON h.security_id = s.security_id
JOIN portfolio_totals pt ON h.portfolio_id = pt.portfolio_id AND h.as_of_date = pt.as_of_date;

SELECT COUNT(*) AS intermediate_rows FROM SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS;

### 3c. Mart Layer (Tables)

The mart layer is where everything comes together. These tables are **business-ready data products** — designed for analysts, dashboards, and regulatory reporting. Every column has a clear business meaning, and the data has been validated and enriched through the layers below.

We will create:

- **`F_POSITIONS_DAILY`** — The core fact table: daily holdings with full security and portfolio context. This is what a portfolio manager queries to see "what do we hold?"
- **`F_PERFORMANCE_DAILY`** — Daily portfolio returns vs benchmark. Essential for performance attribution and client reporting.
- **`D_PORTFOLIO`** and **`D_CLIENT`** — Dimension tables for filtering and grouping.

> **Reference**: [CREATE TABLE AS SELECT](https://docs.snowflake.com/en/sql-reference/sql/create-table)

In [ ]:
USE SCHEMA SNOWCAMP_LAB.MARTS;

-- Fact: Daily Positions
CREATE OR REPLACE TABLE F_POSITIONS_DAILY AS
SELECT
    p.holding_id, p.portfolio_id, po.client_id, po.portfolio_name, po.portfolio_type,
    po.benchmark_id, p.security_id, p.ticker, p.isin, p.security_name,
    p.sector, p.asset_class, p.currency, p.as_of_date,
    p.quantity, p.market_value, p.unit_price, p.total_portfolio_value, p.portfolio_weight
FROM SNOWCAMP_LAB.ANALYTICS.INT_PORTFOLIO_POSITIONS p
JOIN SNOWCAMP_LAB.RAW.PORTFOLIOS po ON p.portfolio_id = po.portfolio_id;

-- Fact: Daily Performance (portfolio return vs benchmark)
CREATE OR REPLACE TABLE F_PERFORMANCE_DAILY AS
WITH portfolio_values AS (
    SELECT portfolio_id, as_of_date, SUM(market_value) AS total_market_value
    FROM SNOWCAMP_LAB.ANALYTICS.STG_HOLDINGS GROUP BY portfolio_id, as_of_date
),
portfolio_returns AS (
    SELECT portfolio_id, as_of_date, total_market_value,
        LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date) AS prev_mv,
        CASE WHEN LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date) > 0
             THEN ROUND((total_market_value - LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date))
                  / LAG(total_market_value) OVER (PARTITION BY portfolio_id ORDER BY as_of_date), 6)
             ELSE 0 END AS daily_return
    FROM portfolio_values
)
SELECT pr.portfolio_id, pf.client_id, pf.benchmark_id, pr.as_of_date, pr.total_market_value,
    pr.daily_return AS portfolio_return, COALESCE(b.daily_return, 0) AS benchmark_return,
    ROUND(pr.daily_return - COALESCE(b.daily_return, 0), 6) AS active_return
FROM portfolio_returns pr
JOIN SNOWCAMP_LAB.RAW.PORTFOLIOS pf ON pr.portfolio_id = pf.portfolio_id
LEFT JOIN SNOWCAMP_LAB.RAW.BENCHMARKS b ON pf.benchmark_id = b.benchmark_id AND pr.as_of_date = b.as_of_date
WHERE pr.prev_mv IS NOT NULL;

-- Dimension: Portfolio
CREATE OR REPLACE TABLE D_PORTFOLIO AS
SELECT p.portfolio_id, p.portfolio_name, p.portfolio_type, p.benchmark_id, p.inception_date,
       p.client_id, c.client_name, c.region, c.aum_tier
FROM SNOWCAMP_LAB.RAW.PORTFOLIOS p
JOIN SNOWCAMP_LAB.RAW.CLIENTS c ON p.client_id = c.client_id;

-- Dimension: Client
CREATE OR REPLACE TABLE D_CLIENT AS
SELECT client_id, client_name, region, aum_tier, onboarding_date
FROM SNOWCAMP_LAB.RAW.CLIENTS;

SELECT 'Core mart tables created!' AS status;

### 3d. Marketplace Join: Real NASDAQ Prices + Synthetic Holdings

This is the step that really shows the **Snowflake Marketplace value proposition**. With a single SQL statement, we join our proprietary portfolio data with real-time market prices — no data vendor contract negotiations, no FTP downloads, no CSV wrangling. The Marketplace data sits in a shared database and is accessed via a standard JOIN.

We create `F_HOLDINGS_WITH_MARKET_DATA`, which enriches every holding with:
- The **real closing price** from NASDAQ
- A **mark-to-market value** (quantity × real price)
- The **valuation difference** between our synthetic prices and real market prices

This is exactly the kind of table a fund administrator would use for NAV calculations and client statements.

In [ ]:
-- Fact: Holdings enriched with real NASDAQ prices from Snowflake Marketplace
CREATE OR REPLACE TABLE SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA AS
SELECT
    p.holding_id, p.portfolio_id, p.client_id,
    p.portfolio_name, p.portfolio_type,
    p.security_id, p.ticker, p.security_name, p.sector, p.asset_class, p.currency,
    p.as_of_date, p.quantity,
    p.market_value       AS synthetic_market_value,
    mkt.close_price      AS market_close_price,
    CASE WHEN mkt.close_price IS NOT NULL
         THEN ROUND(p.quantity * mkt.close_price, 2)
         ELSE p.market_value END AS mark_to_market_value,
    CASE WHEN mkt.close_price IS NOT NULL AND p.market_value > 0
         THEN ROUND((p.quantity * mkt.close_price - p.market_value) / p.market_value, 4)
         ELSE 0 END AS valuation_diff_pct
FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY p
LEFT JOIN SNOWCAMP_LAB.ANALYTICS.STG_MARKET_PRICES mkt
    ON p.ticker = mkt.ticker AND p.as_of_date = mkt.price_date;

In [ ]:
-- See real vs synthetic prices side by side
-- The valuation_diff_pct shows how far our synthetic prices are from reality
SELECT ticker, security_name, as_of_date,
       quantity, synthetic_market_value, market_close_price,
       mark_to_market_value, valuation_diff_pct
FROM SNOWCAMP_LAB.MARTS.F_HOLDINGS_WITH_MARKET_DATA
WHERE market_close_price IS NOT NULL
ORDER BY mark_to_market_value DESC
LIMIT 15;

In [ ]:
-- Quick check: AUM by region — this is the kind of query an analyst runs daily
SELECT c.region,
       COUNT(DISTINCT f.portfolio_id) AS portfolios,
       ROUND(SUM(f.market_value), 0) AS total_aum
FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY f
JOIN SNOWCAMP_LAB.MARTS.D_CLIENT c ON f.client_id = c.client_id
WHERE f.as_of_date = (SELECT MAX(as_of_date) FROM SNOWCAMP_LAB.MARTS.F_POSITIONS_DAILY)
GROUP BY c.region
ORDER BY total_aum DESC;

---
## Part 4: dbt Projects — Deploy, Test & Orchestrate (30 min)

We have just built the transformation pipeline manually with SQL. That works for a lab, but in production you need something more: **version control**, **automated testing**, **dependency management**, and **scheduling**. That is exactly what [dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) provide.

In this section we will:
1. Understand how the dbt project in this lab's Git repo is structured
2. Deploy it as a Snowflake-native **dbt Project object**
3. Execute `dbt build` to run all models and tests
4. Set up a **Snowflake Task** to orchestrate the pipeline on a schedule

The key insight is that dbt on Snowflake is **not** an external tool that connects to Snowflake via a driver — it runs **inside** Snowflake as a first-class object, with the same security, governance, and audit trail as any other SQL command.

> **Reference**: [dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/ui-snowsight/dbt) | [Getting Started Tutorial](https://docs.snowflake.com/en/user-guide/tutorials/dbt-projects-on-snowflake-getting-started-tutorial)

### profiles.yml — Connecting dbt to Snowflake

Every dbt project needs a `profiles.yml` that tells dbt which database, schema, warehouse, and role to use. When running dbt inside Snowflake (as opposed to from your laptop), the `account` and `user` fields can be left as placeholders because the project inherits the session context.

Notice the two **targets**: `dev` writes to the `ANALYTICS` schema (for development and testing) while `prod` writes to `MARTS` (for business-ready consumption). This pattern enables a **promotion workflow** — you develop and test in `ANALYTICS`, and when everything passes, you deploy to `MARTS`.

```yaml
snowcamp:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: 'not needed'
      user: 'not needed'
      role: accountadmin
      database: SNOWCAMP_LAB
      schema: ANALYTICS
      warehouse: WH_LAB
    prod:
      type: snowflake
      account: 'not needed'
      user: 'not needed'
      role: accountadmin
      database: SNOWCAMP_LAB
      schema: MARTS
      warehouse: WH_LAB
```

> **Reference**: [Understand dbt project objects](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake-understanding-dbt-project-objects)

### dbt Tests — Automated Data Quality

One of dbt's superpowers is **built-in testing**. Instead of hoping your data is correct, you declare expectations in `schema.yml` files and dbt validates them automatically with every build.

This lab's repo includes tests at every layer:

**Staging tests** (`models/staging/schema.yml`):
- `holding_id` is `not_null` and `unique` — no orphaned or duplicated positions
- `ticker` is `not_null` — every security must have a Marketplace-joinable key
- `transaction_id` is `not_null` and `unique`

**Mart tests** (`models/marts/schema.yml`):
- `portfolio_id` has a `relationships` test to `d_portfolio` — referential integrity
- `client_id` has a `relationships` test to `d_client`

When you run `dbt build`, models are created first, then tests execute. If any test fails, the build stops and you know exactly which data quality rule was violated. For a regulated industry like asset management, this kind of automated validation is essential.

> **Reference**: [dbt Tests Documentation](https://docs.getdbt.com/docs/build/data-tests)

### Connect to the Lab's Git Repository

Before we can deploy the dbt project, Snowflake needs access to the Git repository that contains the code. We create an **API integration** (which tells Snowflake it is allowed to talk to GitHub) and a **Git Repository object** (which points to our specific repo).

Once connected, Snowflake can read any file in the repo — dbt models, Streamlit apps, configuration files, and more. Think of it as mounting a Git repo as a virtual file system inside Snowflake.

In [ ]:
-- Create an API integration for GitHub
CREATE OR REPLACE API INTEGRATION snowcamp_git_api
    API_PROVIDER       = git_https_api
    API_ALLOWED_PREFIXES = ('https://github.com/')
    ENABLED            = TRUE;

-- Create a Git Repository pointing to the lab repo
CREATE OR REPLACE GIT REPOSITORY SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO
    API_INTEGRATION = snowcamp_git_api
    ORIGIN          = 'https://github.com/sfc-gh-epolano/dws-snowcamp-lab.git';

-- Fetch the latest content
ALTER GIT REPOSITORY SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO FETCH;

-- Verify the dbt project files are accessible
LS @SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO/branches/main/dbt/snowcamp_client_reporting/;

### Deploy and Execute the dbt Project

Now we can create a **dbt Project object** in Snowflake. This is a first-class Snowflake object — just like a table or a view — that encapsulates the entire dbt project: models, tests, sources, and configuration.

When we execute `dbt build`, Snowflake:
1. Reads the model SQL files from the Git repo
2. Resolves the dependency graph (DAG)
3. Executes models in the correct order
4. Runs all tests after each model completes
5. Reports the results

> **Reference**: [CREATE DBT PROJECT](https://docs.snowflake.com/en/sql-reference/sql/create-dbt-project) | [EXECUTE DBT PROJECT](https://docs.snowflake.com/en/sql-reference/sql/execute-dbt-project) | [Deploy dbt project objects](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake-deploy)

In [ ]:
-- Deploy the dbt Project object from the Git repository
CREATE OR REPLACE DBT PROJECT SNOWCAMP_LAB.ANALYTICS.SNOWCAMP_CLIENT_REPORTING
    FROM @SNOWCAMP_LAB.GITREPO.SNOWCAMP_GIT_REPO/branches/main/dbt/snowcamp_client_reporting;

-- Execute dbt build — this runs all models + tests in dependency order
EXECUTE DBT PROJECT SNOWCAMP_LAB.ANALYTICS.SNOWCAMP_CLIENT_REPORTING
    ARGS = 'build --target dev';

### Orchestration with Snowflake Tasks

In production, transformations need to run on a schedule — daily, hourly, or whenever new data arrives. Snowflake **Tasks** provide a built-in scheduler so you don't need an external orchestrator like Airflow or Prefect.

A Task is simply a SQL statement that runs on a schedule. By wrapping `EXECUTE DBT PROJECT` inside a Task, we get a fully automated, production-grade pipeline that:
- Runs every hour (or whatever schedule you choose)
- Uses the `WH_LAB` warehouse for compute
- Can be monitored in the Snowsight UI under **Activity → Task History**
- Sends notifications on failure if configured

Let's create and start a Task now. We will schedule it to run every hour, but you could set any CRON schedule (e.g., once a day at 6 AM UTC).

> **Reference**: [Introduction to Tasks](https://docs.snowflake.com/en/user-guide/tasks-intro) | [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

In [ ]:
-- Create a scheduled Task that runs dbt build every hour
CREATE OR REPLACE TASK SNOWCAMP_LAB.ANALYTICS.DBT_HOURLY_BUILD
    WAREHOUSE = WH_LAB
    SCHEDULE  = 'USING CRON 0 * * * * UTC'
AS
    EXECUTE DBT PROJECT SNOWCAMP_LAB.ANALYTICS.SNOWCAMP_CLIENT_REPORTING
        ARGS = 'build --target dev';

-- Resume the Task to activate the schedule
-- (Tasks are created in a SUSPENDED state by default)
ALTER TASK SNOWCAMP_LAB.ANALYTICS.DBT_HOURLY_BUILD RESUME;

-- Verify the Task is running
SHOW TASKS IN SCHEMA SNOWCAMP_LAB.ANALYTICS;

### DAG Visualisation in Snowsight

To see the full dependency graph of your dbt project:

1. Navigate to **Projects** → **Workspaces** in Snowsight
2. Open (or create) a workspace connected to the Git repository
3. Select the **DAG** tab below the workspace editor
4. You will see all models, tests, and sources as a visual graph

The DAG shows how data flows from `RAW` sources through `STG_*` staging views, `INT_*` intermediate tables, and into `F_*` / `D_*` mart tables — including the Marketplace data join.

This visualisation is incredibly useful for understanding dependencies, debugging issues, and communicating your pipeline architecture to stakeholders.

> **Reference**: [Supported dbt commands](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake-supported-commands)

---
## Day 1 Complete!

Excellent work. Let's take stock of what you have built today:

| What You Built | Why It Matters |
|----------------|---------------|
| **3-schema database** (RAW → ANALYTICS → MARTS) | Industry-standard Bronze/Silver/Gold architecture |
| **6 synthetic source tables** with real NASDAQ tickers | Realistic test data that mirrors production feeds |
| **Marketplace integration** with real stock prices | Zero-copy access to third-party data — no ETL |
| **Staging → Intermediate → Mart** transformation pipeline | Clean, tested, traceable data lineage |
| **dbt Project object** with automated tests | Version-controlled, testable transformations |
| **Scheduled Snowflake Task** running dbt builds | Production-ready orchestration |

### What's Coming Tomorrow (Day 2)

In Day 2 we will build on top of everything created today:

1. **Streamlit Dashboard** — Turn the mart tables into an interactive client-reporting app with KPIs, charts, and drill-downs
2. **Horizon Catalog & Internal Marketplace** — Tag data products with business metadata and publish them for internal consumers
3. **Platform Administration** — Monitor credit usage, analyse query performance, and set up resource monitors
4. **Cortex Code** — Explore how Snowflake's AI coding assistant can accelerate your work

> *See you tomorrow! Your data will be waiting for you.* 🎿

---
### Housekeeping: Suspend the Task

To avoid unnecessary credit usage overnight, let's suspend the scheduled Task. We will resume it again tomorrow if needed.

> If you are not continuing to Day 2, see the full teardown at the end of the Day 2 notebook.

In [ ]:
-- Suspend the Task to avoid credit usage while we are not using it
ALTER TASK SNOWCAMP_LAB.ANALYTICS.DBT_HOURLY_BUILD SUSPEND;

SHOW TASKS IN SCHEMA SNOWCAMP_LAB.ANALYTICS;